In [1]:
import os
import cv2
import numpy as np
import joblib
import matplotlib.pyplot as plt
from tqdm import tqdm

test_dir = r"E:\Road_Quality\Frame_Num\MatchedFrames"
model_dir = r"E:\Road_Quality\model\CSI_percentile_linear"

model = joblib.load(os.path.join(model_dir, "model.joblib"))

def list_jpgs(folder):
    return [os.path.join(folder, x) for x in os.listdir(folder) if x.lower().endswith(".jpg")]

def compute_csi(gray):
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return float(np.std(lap))

def compute_lcr(gray):
    edges = cv2.Canny(gray, 100, 200)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=50, minLineLength=30, maxLineGap=10)
    if lines is None:
        return 0.0
    tot = 0.0
    for l in lines:
        x1, y1, x2, y2 = np.ravel(l)[:4]
        tot += float(np.sqrt((x2-x1)**2 + (y2-y1)**2))
    return float(tot / (edges.size + 1e-6))

paths = list_jpgs(test_dir)

X = []
for p in tqdm(paths, desc="Testing CSI-linear on MatchedFrames"):
    g = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if g is None:
        continue
    csi = compute_csi(g)
    lcr = compute_lcr(g)
    mu = float(np.mean(g))
    sd = float(np.std(g))
    X.append([csi, lcr, mu, sd])

X = np.array(X, dtype=np.float32)

pred = model.predict(X)
pred = np.clip(pred, 0.0, 1.0)

# Plot distribution (bar chart bins)
bins = np.linspace(0, 1.0, 11)
hist, edges = np.histogram(pred, bins=bins)
centers = 0.5 * (edges[:-1] + edges[1:])

plt.figure()
plt.bar(centers, hist, width=(edges[1]-edges[0]) * 0.9)
plt.xlabel("Predicted CSI Percentile (0..1)")
plt.ylabel("Count")
plt.title("CSI Percentile Predictions (Linear Model) on MatchedFrames")
plt.tight_layout()
plt.show()

print("CSI predicted percentile quartiles:", np.quantile(pred, [0, 0.25, 0.5, 0.75, 1.0]))


C:\Users\Shaif\anaconda3\envs\dl\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator Ridge from version 1.3.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Testing CSI-linear on MatchedFrames:   1%|▏                                          | 18/3339 [00:02<07:18,  7.57it/s]


KeyboardInterrupt: 

In [3]:
import os
import cv2
import numpy as np
import joblib
from tqdm import tqdm

ranked_dir = r"E:\Road_Quality\RankedFrames2"

csi_model = joblib.load(r"E:\Road_Quality\model\CSI_percentile_linear\model.joblib")
lcr_model = joblib.load(r"E:\Road_Quality\model\LCR_percentile_linear\model.joblib")

def compute_csi(gray):
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return float(np.std(lap))

def compute_lcr(gray):
    edges = cv2.Canny(gray, 100, 200)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=50, minLineLength=30, maxLineGap=10)
    if lines is None:
        return 0.0
    tot = 0.0
    for l in lines:
        x1, y1, x2, y2 = np.ravel(l)[:4]
        tot += float(np.sqrt((x2-x1)**2 + (y2-y1)**2))
    return float(tot / (edges.size + 1e-6))

ranked_paths = [os.path.join(ranked_dir, x) for x in os.listdir(ranked_dir) if x.lower().endswith((".jpg", ".jpeg", ".png"))]
print("Found", len(ranked_paths), "images in", ranked_dir)

results = []
for p in tqdm(ranked_paths, desc="Predicting on RankedFrames"):
    g = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if g is None:
        continue
    csi = compute_csi(g)
    lcr = compute_lcr(g)
    mu = float(np.mean(g))
    sd = float(np.std(g))
    feat = np.array([[csi, lcr, mu, sd]], dtype=np.float32)
    csi_pred = float(np.clip(csi_model.predict(feat)[0], 0.0, 1.0))
    lcr_pred = float(np.clip(lcr_model.predict(feat)[0], 0.0, 1.0))
    results.append((os.path.basename(p), csi_pred, lcr_pred))

for name, csi_pred, lcr_pred in results:
    print(name, "-> CSI percentile:", round(csi_pred, 4), "| LCR percentile:", round(lcr_pred, 4))

C:\Users\Shaif\anaconda3\envs\dl\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator Ridge from version 1.3.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Found 50 images in E:\Road_Quality\RankedFrames2


Predicting on RankedFrames: 100%|██████████████████████████████████████████████████████| 50/50 [00:01<00:00, 33.97it/s]

1.jpg -> CSI percentile: 0.9986 | LCR percentile: 1.0
10.jpg -> CSI percentile: 0.7545 | LCR percentile: 0.8529
11.jpg -> CSI percentile: 0.716 | LCR percentile: 0.8308
12.jpg -> CSI percentile: 0.8773 | LCR percentile: 0.5717
13.jpg -> CSI percentile: 0.9294 | LCR percentile: 0.6027
14.jpg -> CSI percentile: 0.8722 | LCR percentile: 0.5756
15.jpg -> CSI percentile: 0.8717 | LCR percentile: 0.9094
16.jpg -> CSI percentile: 0.7363 | LCR percentile: 0.848
17.jpg -> CSI percentile: 0.752 | LCR percentile: 0.869
18.jpg -> CSI percentile: 0.8486 | LCR percentile: 0.4823
19.jpg -> CSI percentile: 0.8916 | LCR percentile: 0.5163
2.jpg -> CSI percentile: 0.937 | LCR percentile: 1.0
20.jpg -> CSI percentile: 0.7767 | LCR percentile: 0.6288
21.jpg -> CSI percentile: 0.7926 | LCR percentile: 0.6702
22.jpg -> CSI percentile: 0.929 | LCR percentile: 0.6782
23.jpg -> CSI percentile: 0.8324 | LCR percentile: 0.8836
24.jpg -> CSI percentile: 0.8664 | LCR percentile: 0.6046
25.jpg -> CSI percentile: 0.

In [7]:
import skimage, scipy, cv2; print(skimage.__version__, scipy.__version__, cv2.__version__)

0.25.2 1.15.3 5.0.0


In [4]:
exec('import os\nimport numpy as np\nimport cv2\nfrom scipy.stats import spearmanr, kendalltau\n\nranked_dir = r"E:\\Road_Quality\\RankedFrames2"\n\ndef compute_psnr(a, b):\n    a = a.astype(np.float64)\n    b = b.astype(np.float64)\n    mse = np.mean((a - b) ** 2)\n    if mse == 0:\n        return 100.0\n    return 20 * np.log10(255.0 / np.sqrt(mse))\n\ndef compute_ssim(a, b):\n    a = a.astype(np.float64)\n    b = b.astype(np.float64)\n    C1 = (0.01 * 255) ** 2\n    C2 = (0.03 * 255) ** 2\n    kernel = cv2.getGaussianKernel(11, 1.5)\n    window = np.outer(kernel, kernel.transpose())\n    mu1 = cv2.filter2D(a, -1, window)\n    mu2 = cv2.filter2D(b, -1, window)\n    mu1_sq = mu1 * mu1\n    mu2_sq = mu2 * mu2\n    mu1_mu2 = mu1 * mu2\n    sigma1_sq = cv2.filter2D(a * a, -1, window) - mu1_sq\n    sigma2_sq = cv2.filter2D(b * b, -1, window) - mu2_sq\n    sigma12 = cv2.filter2D(a * b, -1, window) - mu1_mu2\n    num = (2 * mu1_mu2 + C1) * (2 * sigma12 + C2)\n    den = (mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2)\n    return float(np.mean(num / den))\n\ndef numeric_key(name):\n    return int(os.path.splitext(name)[0])\n\nresults_dict = {}\nfor name, c, l in results:\n    results_dict[name] = (c, l)\n\nnames_sorted = sorted(results_dict.keys(), key=numeric_key)\nref_rank = {}\nfor i, name in enumerate(names_sorted):\n    ref_rank[name] = i + 1\nref_quality = {}\nfor name in names_sorted:\n    ref_quality[name] = 11 - ref_rank[name]\n\nimgs = {}\nfor name in names_sorted:\n    g = cv2.imread(os.path.join(ranked_dir, name), cv2.IMREAD_GRAYSCALE)\n    imgs[name] = g\n\nref_name = names_sorted[0]\nref_img = imgs[ref_name]\nrh, rw = ref_img.shape\n\nssim_scores = {}\npsnr_scores = {}\nfor name in names_sorted:\n    g_r = cv2.resize(imgs[name], (rw, rh))\n    ssim_scores[name] = compute_ssim(ref_img, g_r)\n    psnr_scores[name] = compute_psnr(ref_img, g_r)\n\nprint("Reference frame used for SSIM/PSNR:", ref_name)\nprint("Frame".ljust(8) + "NameRank".ljust(10) + "CSI".ljust(10) + "LCR".ljust(10) + "SSIM".ljust(10) + "PSNR".ljust(10))\nfor name in names_sorted:\n    csi_v, lcr_v = results_dict[name]\n    row = name.ljust(8) + str(ref_rank[name]).ljust(10) + ("%.4f" % csi_v).ljust(10) + ("%.4f" % lcr_v).ljust(10) + ("%.4f" % ssim_scores[name]).ljust(10) + ("%.2f" % psnr_scores[name]).ljust(10)\n    print(row)\n\nref_q_vals = [ref_quality[n] for n in names_sorted]\nmethods = {}\nmethods["CSI"] = [results_dict[n][0] for n in names_sorted]\nmethods["LCR"] = [results_dict[n][1] for n in names_sorted]\nmethods["SSIM"] = [ssim_scores[n] for n in names_sorted]\nmethods["PSNR"] = [psnr_scores[n] for n in names_sorted]\n\norder = ["CSI", "LCR", "SSIM", "PSNR"]\nspearman_row = []\nkendall_row = []\nfor m in order:\n    vals = methods[m]\n    rho, _ = spearmanr(ref_q_vals, vals)\n    tau, _ = kendalltau(ref_q_vals, vals)\n    spearman_row.append(rho)\n    kendall_row.append(tau)\n\nprint("")\nprint("Ranking-agreement scores (vs. filename-order reference ranking, 1.jpg=best .. 10.jpg=worst):")\nprint("Metric".ljust(10) + "CSI".rjust(10) + "LCR".rjust(10) + "SSIM".rjust(10) + "PSNR".rjust(10))\nprint("Spearman".ljust(10) + ("%.4f" % spearman_row[0]).rjust(10) + ("%.4f" % spearman_row[1]).rjust(10) + ("%.4f" % spearman_row[2]).rjust(10) + ("%.4f" % spearman_row[3]).rjust(10))\nprint("Kendall".ljust(10) + ("%.4f" % kendall_row[0]).rjust(10) + ("%.4f" % kendall_row[1]).rjust(10) + ("%.4f" % kendall_row[2]).rjust(10) + ("%.4f" % kendall_row[3]).rjust(10))\n')

Reference frame used for SSIM/PSNR: 1.jpg
Frame   NameRank  CSI       LCR       SSIM      PSNR      
1.jpg   1         0.9986    1.0000    1.0000    100.00    
2.jpg   2         0.9370    1.0000    0.9051    20.57     
3.jpg   3         0.9295    0.9577    0.9207    21.05     
4.jpg   4         0.9657    0.9862    0.9245    21.13     
5.jpg   5         0.7296    0.8328    0.9301    21.46     
6.jpg   6         0.7070    0.8226    0.9308    21.47     
7.jpg   7         0.7591    0.8636    0.9305    21.57     
8.jpg   8         0.6941    0.8205    0.9319    21.90     
9.jpg   9         0.7379    0.8532    0.9320    21.69     
10.jpg  10        0.7545    0.8529    0.9328    21.62     
11.jpg  11        0.7160    0.8308    0.9341    21.77     
12.jpg  12        0.8773    0.5717    0.9326    20.75     
13.jpg  13        0.9294    0.6027    0.9388    21.61     
14.jpg  14        0.8722    0.5756    0.9346    21.35     
15.jpg  15        0.8717    0.9094    0.9449    23.61     
16.jpg  16    

In [5]:
exec("import importlib\nfor pkg in ['torch','lpips','brisque','skimage','scipy']:\n    try:\n        m = importlib.import_module(pkg)\n        print(pkg, getattr(m, '__version__', 'unknown'))\n    except Exception as e:\n        print(pkg, 'NOT AVAILABLE:', e)\ntry:\n    import pyiqa\n    print('pyiqa', pyiqa.__version__)\nexcept Exception as e:\n    print('pyiqa NOT AVAILABLE:', e)\n")

torch 2.5.1+cu121
lpips NOT AVAILABLE: No module named 'lpips'
brisque NOT AVAILABLE: No module named 'brisque'
skimage 0.25.2
scipy 1.15.3
pyiqa NOT AVAILABLE: No module named 'pyiqa'


In [6]:
!pip install pyiqa --quiet

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Shaif\\anaconda3\\envs\\dl\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.



In [7]:
!pip install pyiqa --no-deps --quiet

In [8]:
import pyiqa; print(pyiqa.__version__)

C:\Users\Shaif\anaconda3\envs\dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0.1.16


In [9]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
brisque_metric = pyiqa.create_metric('brisque', device=device)
print('brisque loaded')
niqe_metric = pyiqa.create_metric('niqe', device=device)
print('niqe loaded')
lpips_metric = pyiqa.create_metric('lpips', device=device)
print('lpips loaded')

device: cuda
Downloading: "https://huggingface.co/chaofengc/IQA-PyTorch-Weights/resolve/main/brisque_svm_weights.pth" to C:\Users\Shaif\.cache\torch\hub\pyiqa\brisque_svm_weights.pth



100%|███████████████████████████████████████████████████████████████████████████████| 112k/112k [00:00<00:00, 2.90MB/s]


brisque loaded
Downloading: "https://huggingface.co/chaofengc/IQA-PyTorch-Weights/resolve/main/niqe_modelparameters.mat" to C:\Users\Shaif\.cache\torch\hub\pyiqa\niqe_modelparameters.mat



100%|█████████████████████████████████████████████████████████████████████████████| 8.15k/8.15k [00:00<00:00, 2.82MB/s]


niqe loaded


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to C:\Users\Shaif/.cache\torch\hub\checkpoints\alexnet-owt-7be5be79.pth
100%|███████████████████████████████████████████████████████████████████████████████| 233M/233M [00:09<00:00, 25.8MB/s]


Downloading: "https://huggingface.co/chaofengc/IQA-PyTorch-Weights/resolve/main/LPIPS_v0.1_alex-df73285e.pth" to C:\Users\Shaif\.cache\torch\hub\pyiqa\LPIPS_v0.1_alex-df73285e.pth



100%|█████████████████████████████████████████████████████████████████████████████| 5.87k/5.87k [00:00<00:00, 2.96MB/s]

Loading pretrained model LPIPS from C:\Users\Shaif\.cache\torch\hub\pyiqa\LPIPS_v0.1_alex-df73285e.pth
lpips loaded


In [10]:
exec("import time\nt0 = time.time()\nref_path = os.path.join(ranked_dir, ref_name)\nbrisque_scores = {}\nniqe_scores = {}\nlpips_scores = {}\nfor name in names_sorted:\n    p = os.path.join(ranked_dir, name)\n    try:\n        brisque_scores[name] = float(brisque_metric(p).item())\n    except Exception as e:\n        brisque_scores[name] = float('nan')\n    try:\n        niqe_scores[name] = float(niqe_metric(p).item())\n    except Exception as e:\n        niqe_scores[name] = float('nan')\n    try:\n        lpips_scores[name] = float(lpips_metric(p, ref_path).item())\n    except Exception as e:\n        lpips_scores[name] = float('nan')\nprint('done computing BRISQUE/NIQE/LPIPS for', len(names_sorted), 'images in', round(time.time()-t0,1), 'sec')\n")

done computing BRISQUE/NIQE/LPIPS for 50 images in 15.1 sec


In [11]:
exec("order2 = ['CSI','LCR','SSIM','PSNR','BRISQUE','NIQE','LPIPS']\nmethods2 = dict(methods)\nmethods2['BRISQUE'] = [brisque_scores[n] for n in names_sorted]\nmethods2['NIQE'] = [niqe_scores[n] for n in names_sorted]\nmethods2['LPIPS'] = [lpips_scores[n] for n in names_sorted]\nspearman_row2 = []\nkendall_row2 = []\nfor m in order2:\n    vals = methods2[m]\n    rho, _ = spearmanr(ref_q_vals, vals)\n    tau, _ = kendalltau(ref_q_vals, vals)\n    spearman_row2.append(rho)\n    kendall_row2.append(tau)\nprint('Full per-image table (Rank 1 = best per filename-order reference):')\nheader = 'Frame'.ljust(8) + 'Rank'.ljust(6) + 'CSI'.ljust(8) + 'LCR'.ljust(8) + 'SSIM'.ljust(8) + 'PSNR'.ljust(8) + 'BRISQUE'.ljust(10) + 'NIQE'.ljust(8) + 'LPIPS'.ljust(8)\nprint(header)\nfor name in names_sorted:\n    row = name.ljust(8) + str(ref_rank[name]).ljust(6) + ('%.4f' % results_dict[name][0]).ljust(8) + ('%.4f' % results_dict[name][1]).ljust(8) + ('%.4f' % ssim_scores[name]).ljust(8) + ('%.2f' % psnr_scores[name]).ljust(8) + ('%.4f' % brisque_scores[name]).ljust(10) + ('%.4f' % niqe_scores[name]).ljust(8) + ('%.4f' % lpips_scores[name]).ljust(8)\n    print(row)\nprint('')\nprint('Ranking-agreement scores (vs. filename-order reference ranking, 1.jpg=best .. 50.jpg=worst):')\nhdr2 = 'Metric'.ljust(10)\nfor m in order2:\n    hdr2 += m.rjust(10)\nprint(hdr2)\nsrow = 'Spearman'.ljust(10)\nfor v in spearman_row2:\n    srow += ('%.4f' % v).rjust(10)\nprint(srow)\nkrow = 'Kendall'.ljust(10)\nfor v in kendall_row2:\n    krow += ('%.4f' % v).rjust(10)\nprint(krow)\n")

Full per-image table (Rank 1 = best per filename-order reference):
Frame   Rank  CSI     LCR     SSIM    PSNR    BRISQUE   NIQE    LPIPS   
1.jpg   1     0.9986  1.0000  1.0000  100.00  93.5427   18.1842 0.0000  
2.jpg   2     0.9370  1.0000  0.9051  20.57   89.9106   13.3141 0.1590  
3.jpg   3     0.9295  0.9577  0.9207  21.05   92.5539   17.3164 0.1683  
4.jpg   4     0.9657  0.9862  0.9245  21.13   92.4919   14.9698 0.1317  
5.jpg   5     0.7296  0.8328  0.9301  21.46   92.4826   18.2252 0.1045  
6.jpg   6     0.7070  0.8226  0.9308  21.47   92.3801   20.5380 0.1074  
7.jpg   7     0.7591  0.8636  0.9305  21.57   93.1442   20.7833 0.1121  
8.jpg   8     0.6941  0.8205  0.9319  21.90   96.0724   17.0261 0.1363  
9.jpg   9     0.7379  0.8532  0.9320  21.69   91.9211   20.4704 0.1054  
10.jpg  10    0.7545  0.8529  0.9328  21.62   92.7580   20.6677 0.1034  
11.jpg  11    0.7160  0.8308  0.9341  21.77   93.3459   21.1924 0.1007  
12.jpg  12    0.8773  0.5717  0.9326  20.75   89.8493   2